# CarbonTatvaAI: KPI-to-ESG Summary Fine-Tuning

**Task:** company metadata + structured BRSR KPIs → one factual ESG narrative summary.

This notebook:

1. Uses only the BRSR 2024-25 and 2025-26 PRD master datasets.
2. Builds clean company-level targets in the approved ADF Foods/Alicon style.
3. Creates company-disjoint train, validation, and test splits.
4. Fine-tunes Llama 3.1 8B Instruct with Unsloth 4-bit QLoRA.
5. Saves resumable checkpoints every 15 minutes.
6. Evaluates KPI fidelity on unseen companies.

It does not use SusGen, raw PDF text, RAG, intent labels, section-classification labels, or the old `llm_training_summary` audit field.


In [ ]:
# Cell 1: Install one compatible Unsloth stack
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "unsloth")
if importlib.util.find_spec("rouge_score") is None:
    pip_install("rouge-score")
if importlib.util.find_spec("bert_score") is None:
    pip_install("bert-score")

print("Dependencies ready. Do not separately upgrade transformers, trl, or datasets after this cell.")


In [ ]:
# Cell 2: Paths and run configuration
import os
import json
import shutil
import time
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working/carbontatva_kpi_summary") if IS_KAGGLE else Path.cwd() / "carbontatva_kpi_summary"
DATA_OUTPUT_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
FINAL_ADAPTER_DIR = WORK_DIR / "final_lora_adapter"
ARTIFACT_DIR = WORK_DIR / "artifacts"

for directory in (WORK_DIR, DATA_OUTPUT_DIR, CHECKPOINT_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
SAVE_EVERY_MINUTES = 15
SAVE_STEPS = 50
MAX_EVAL_SAMPLES = 30
RUN_BERTSCORE = True
SEED = 3407

# Set to True only for a quick pipeline smoke test.
SMOKE_TEST = False
SMOKE_TRAIN_EXAMPLES = 120
SMOKE_MAX_STEPS = 10

print("Work directory:", WORK_DIR)
print("Kaggle:", IS_KAGGLE)


## Dataset preparation

The following cell contains the complete CPU-only dataset builder, so the notebook can run from either:

- the provided prebuilt `kpi_summary_*.jsonl` files, or
- the two raw PRD master CSV files plus the optional compact company-name map.


In [ ]:
#!/usr/bin/env python3
"""Build company-level KPI-to-ESG-summary fine-tuning data.

The source PRD tables contain extracted metadata, KPIs, evidence, intent labels,
and an ``llm_training_summary`` audit field. This builder intentionally uses
only company metadata and scalar KPI columns. It creates a fresh, factual
narrative target for each company and never trains on the audit summary.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable


SYSTEM_PROMPT = (
    "You are CarbonTatvaAI, an ESG reporting analyst. Write a concise, factual "
    "ESG narrative summary using only the supplied company metadata and KPI "
    "data. Mention all material KPI values that are provided, preserve units, "
    "and describe year-on-year movement accurately. Do not invent policies, "
    "initiatives, awards, targets, committees, certifications, framework "
    "alignment, or explanations that are absent from the input. The summary is "
    "a drafting aid, not a complete statutory BRSR report."
)

META_FIELDS = {
    "company": "Company",
    "reporting_year": "Reporting year",
    "meta_sector": "Sector",
    "meta_market_cap": "Market capitalisation",
    "meta_framework_used": "Reporting framework",
    "meta_brsr_version": "BRSR version",
    "meta_assurance_type": "Assurance",
    "meta_geography": "Geography",
}

EXCLUDED_KPI_SUFFIXES = (
    "_evidence",
    "_source",
    "_json",
)

NUMERIC_KPI_FIELDS = [
    "kpi_scope1_emissions_tco2e_current",
    "kpi_scope1_emissions_tco2e_previous",
    "kpi_scope1_emissions_yoy_reduction_percent",
    "kpi_scope2_emissions_tco2e_current",
    "kpi_scope2_emissions_tco2e_previous",
    "kpi_scope2_emissions_yoy_reduction_percent",
    "kpi_scope3_emissions_tco2e_current",
    "kpi_scope3_emissions_tco2e_previous",
    "kpi_scope3_emissions_yoy_reduction_percent",
    "kpi_scope1_scope2_total_tco2e_current",
    "kpi_scope1_scope2_total_tco2e_previous",
    "kpi_scope1_scope2_yoy_reduction_percent",
    "kpi_renewable_energy_percent",
    "kpi_renewable_energy_consumption_gj",
    "kpi_total_energy_consumption_gj",
    "kpi_water_consumption_kl_current",
    "kpi_water_consumption_kl_previous",
    "kpi_water_consumption_yoy_reduction_percent",
    "kpi_water_withdrawal_kl_current",
    "kpi_water_withdrawal_kl_previous",
    "kpi_water_withdrawal_yoy_reduction_percent",
    "kpi_total_waste_generated_current",
    "kpi_total_waste_generated_previous",
    "kpi_waste_recycled_current",
    "kpi_waste_recycled_previous",
    "kpi_waste_recycled_percent",
    "kpi_female_employee_percent",
    "kpi_women_on_board_percent",
    "kpi_energy_intensity_current",
    "kpi_energy_intensity_previous",
    "kpi_energy_intensity_yoy_reduction_percent",
]

TEXT_KPI_FIELDS = [
    "kpi_waste_recycled_unit",
    "kpi_energy_intensity_unit",
    "kpi_net_zero_target_year",
]

PERCENTAGE_LEVEL_FIELDS = {
    "kpi_renewable_energy_percent",
    "kpi_waste_recycled_percent",
    "kpi_female_employee_percent",
    "kpi_women_on_board_percent",
}

YOY_SPECS = {
    "scope1": (
        "kpi_scope1_emissions_tco2e_current",
        "kpi_scope1_emissions_tco2e_previous",
        "kpi_scope1_emissions_yoy_reduction_percent",
    ),
    "scope2": (
        "kpi_scope2_emissions_tco2e_current",
        "kpi_scope2_emissions_tco2e_previous",
        "kpi_scope2_emissions_yoy_reduction_percent",
    ),
    "scope3": (
        "kpi_scope3_emissions_tco2e_current",
        "kpi_scope3_emissions_tco2e_previous",
        "kpi_scope3_emissions_yoy_reduction_percent",
    ),
    "scope1_scope2_total": (
        "kpi_scope1_scope2_total_tco2e_current",
        "kpi_scope1_scope2_total_tco2e_previous",
        "kpi_scope1_scope2_yoy_reduction_percent",
    ),
    "water_consumption": (
        "kpi_water_consumption_kl_current",
        "kpi_water_consumption_kl_previous",
        "kpi_water_consumption_yoy_reduction_percent",
    ),
    "water_withdrawal": (
        "kpi_water_withdrawal_kl_current",
        "kpi_water_withdrawal_kl_previous",
        "kpi_water_withdrawal_yoy_reduction_percent",
    ),
    "energy_intensity": (
        "kpi_energy_intensity_current",
        "kpi_energy_intensity_previous",
        "kpi_energy_intensity_yoy_reduction_percent",
    ),
}

FIELD_LABELS = {
    "kpi_scope1_emissions_tco2e_current": "Scope 1 emissions, current year (tCO2e)",
    "kpi_scope1_emissions_tco2e_previous": "Scope 1 emissions, previous year (tCO2e)",
    "kpi_scope1_emissions_yoy_reduction_percent": "Scope 1 year-on-year reduction convention (%)",
    "kpi_scope2_emissions_tco2e_current": "Scope 2 emissions, current year (tCO2e)",
    "kpi_scope2_emissions_tco2e_previous": "Scope 2 emissions, previous year (tCO2e)",
    "kpi_scope2_emissions_yoy_reduction_percent": "Scope 2 year-on-year reduction convention (%)",
    "kpi_scope3_emissions_tco2e_current": "Scope 3 emissions, current year (tCO2e)",
    "kpi_scope3_emissions_tco2e_previous": "Scope 3 emissions, previous year (tCO2e)",
    "kpi_scope3_emissions_yoy_reduction_percent": "Scope 3 year-on-year reduction convention (%)",
    "kpi_scope1_scope2_total_tco2e_current": "Combined Scope 1 and Scope 2 emissions, current year (tCO2e)",
    "kpi_scope1_scope2_total_tco2e_previous": "Combined Scope 1 and Scope 2 emissions, previous year (tCO2e)",
    "kpi_scope1_scope2_yoy_reduction_percent": "Combined Scope 1 and Scope 2 year-on-year reduction convention (%)",
    "kpi_renewable_energy_percent": "Renewable energy share (%)",
    "kpi_renewable_energy_consumption_gj": "Renewable energy consumption (GJ)",
    "kpi_total_energy_consumption_gj": "Total energy consumption (GJ)",
    "kpi_water_consumption_kl_current": "Water consumption, current year (KL)",
    "kpi_water_consumption_kl_previous": "Water consumption, previous year (KL)",
    "kpi_water_consumption_yoy_reduction_percent": "Water consumption year-on-year reduction convention (%)",
    "kpi_water_withdrawal_kl_current": "Water withdrawal, current year (KL)",
    "kpi_water_withdrawal_kl_previous": "Water withdrawal, previous year (KL)",
    "kpi_water_withdrawal_yoy_reduction_percent": "Water withdrawal year-on-year reduction convention (%)",
    "kpi_total_waste_generated_current": "Total waste generated, current year",
    "kpi_total_waste_generated_previous": "Total waste generated, previous year",
    "kpi_waste_recycled_current": "Waste recycled or recovered, current year",
    "kpi_waste_recycled_previous": "Waste recycled or recovered, previous year",
    "kpi_waste_recycled_unit": "Waste recycled or recovered unit",
    "kpi_waste_recycled_percent": "Waste recycled or recovered share (%)",
    "kpi_female_employee_percent": "Female employees (%)",
    "kpi_women_on_board_percent": "Women on the Board (%)",
    "kpi_energy_intensity_current": "Energy intensity, current year",
    "kpi_energy_intensity_previous": "Energy intensity, previous year",
    "kpi_energy_intensity_unit": "Energy intensity unit",
    "kpi_energy_intensity_yoy_reduction_percent": "Energy intensity year-on-year reduction convention (%)",
    "kpi_net_zero_target_year": "Disclosed net-zero target year",
}


@dataclass
class SourceRow:
    row: dict[str, Any]
    source_file: str
    source_dataset: str
    company: str
    company_key: str
    nse_symbol: str


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).replace("\ufeff", "").replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


def is_missing(value: Any) -> bool:
    return clean_text(value).lower() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
        "unknown",
        "not mentioned",
        "not available",
        "not disclosed",
        "[]",
        "{}",
    }


def as_number(value: Any) -> float | None:
    if is_missing(value):
        return None
    text = clean_text(value).replace(",", "").replace("%", "")
    try:
        number = float(text)
    except ValueError:
        return None
    return number if math.isfinite(number) else None


def format_number(value: Any) -> str | None:
    number = as_number(value)
    if number is None:
        return None
    if abs(number) != 0 and abs(number) < 0.01:
        return f"{number:.8f}".rstrip("0").rstrip(".")
    if abs(number - round(number)) < 1e-9:
        return f"{int(round(number)):,}"
    return f"{number:,.2f}".rstrip("0").rstrip(".")


def format_percent(value: Any) -> str | None:
    number = as_number(value)
    if number is None:
        return None
    return f"{abs(number):.2f}".rstrip("0").rstrip(".") + "%"


def normalize_source_key(value: Any) -> str:
    text = clean_text(value).lower()
    text = re.sub(r"\.pdf$", "", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def canonical_company_key(value: Any) -> str:
    text = clean_text(value).upper()
    text = re.sub(
        r"\b(THE|LIMITED|LTD|PRIVATE|PVT|INDIA|COMPANY|CO|CORPORATION|CORP)\b",
        " ",
        text,
    )
    return re.sub(r"[^A-Z0-9]+", "", text)


def title_company_name(value: Any) -> str:
    text = clean_text(value)
    if not text:
        return ""
    if "_" in text or re.search(r"\d{6,}", text):
        first = re.split(r"[_\s-]+", text)[0]
        return first.upper() if first else text
    if text.isupper():
        small_words = {"and", "of", "the", "for", "in"}
        words = []
        for index, word in enumerate(text.lower().split()):
            words.append(word if index and word in small_words else word.capitalize())
        return " ".join(words)
    return text


def normalize_reporting_year(value: Any, default: str = "FY 2024-25") -> str:
    text = clean_text(value).replace("\u2013", "-").replace("\u2014", "-")
    match = re.search(r"(20\d{2})\s*[-/]\s*(\d{2})", text)
    if match:
        first = int(match.group(1))
        second = int("20" + match.group(2))
        if second == first + 1 and 2020 <= first <= 2030:
            return f"FY {first}-{str(second)[-2:]}"
    return default


def source_label(path: Path) -> str:
    name = path.name.lower()
    if "24_25" in name or "24-25" in name:
        return "BRSR 2024-25 source collection"
    if "25_26" in name or "25-26" in name:
        return "BRSR 2025-26 source collection"
    return path.stem


def load_company_map(path: Path | None) -> dict[str, dict[str, str]]:
    if path is None:
        return {}
    mapping: dict[str, dict[str, str]] = {}
    if path.suffix.lower() == ".json":
        raw = json.loads(path.read_text(encoding="utf-8"))
        for source_key, item in raw.items():
            company = clean_text(item.get("company"))
            if source_key and company:
                mapping[normalize_source_key(source_key)] = {
                    "company": company,
                    "nse_symbol": clean_text(item.get("nse_symbol")),
                }
        return mapping
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if not line.strip():
                continue
            item = json.loads(line)
            source_key = normalize_source_key(item.get("source_key"))
            company = clean_text(item.get("company"))
            if source_key and company:
                mapping[source_key] = {
                    "company": company,
                    "nse_symbol": clean_text(item.get("nse_symbol")),
                }
    return mapping


def find_csvs(paths: Iterable[str]) -> list[Path]:
    found: list[Path] = []
    for raw in paths:
        path = Path(raw)
        if path.is_file() and path.suffix.lower() == ".csv":
            found.append(path)
        elif path.is_dir():
            found.extend(
                item
                for item in path.rglob("*.csv")
                if "audit" not in item.name.lower()
                and ("24_25" in item.name or "24-25" in item.name or "25-26" in item.name or "25_26" in item.name)
            )
    return sorted(set(found))


def valid_kpi_value(field: str, value: Any) -> float | str | None:
    if field in TEXT_KPI_FIELDS:
        text = clean_text(value)
        if is_missing(text):
            return None
        if field == "kpi_net_zero_target_year":
            match = re.search(r"\b(20\d{2})\b", text)
            if not match or not 2025 <= int(match.group(1)) <= 2100:
                return None
            return match.group(1)
        return text[:160]

    number = as_number(value)
    if number is None:
        return None
    if field in PERCENTAGE_LEVEL_FIELDS and not 0 <= number <= 100:
        return None
    if "_yoy_reduction_percent" in field:
        return number if abs(number) <= 10000 else None
    return number if number >= 0 else None


def compute_yoy(current: Any, previous: Any) -> float | None:
    current_number = as_number(current)
    previous_number = as_number(previous)
    if current_number is None or previous_number in {None, 0}:
        return None
    return (previous_number - current_number) / abs(previous_number) * 100


def clean_kpis(row: dict[str, Any]) -> dict[str, Any]:
    kpis: dict[str, Any] = {}
    for field in NUMERIC_KPI_FIELDS + TEXT_KPI_FIELDS:
        value = valid_kpi_value(field, row.get(field))
        if value is not None:
            kpis[field] = value

    scope1_current = kpis.get("kpi_scope1_emissions_tco2e_current")
    scope2_current = kpis.get("kpi_scope2_emissions_tco2e_current")
    scope1_previous = kpis.get("kpi_scope1_emissions_tco2e_previous")
    scope2_previous = kpis.get("kpi_scope2_emissions_tco2e_previous")
    if "kpi_scope1_scope2_total_tco2e_current" not in kpis and scope1_current is not None and scope2_current is not None:
        kpis["kpi_scope1_scope2_total_tco2e_current"] = scope1_current + scope2_current
    if "kpi_scope1_scope2_total_tco2e_previous" not in kpis and scope1_previous is not None and scope2_previous is not None:
        kpis["kpi_scope1_scope2_total_tco2e_previous"] = scope1_previous + scope2_previous

    for _, (current_field, previous_field, yoy_field) in YOY_SPECS.items():
        calculated = compute_yoy(kpis.get(current_field), kpis.get(previous_field))
        if calculated is not None:
            kpis[yoy_field] = calculated
        elif yoy_field in kpis and (
            current_field not in kpis or previous_field not in kpis
        ):
            kpis.pop(yoy_field, None)

    return kpis


def row_quality(kpis: dict[str, Any], metadata: dict[str, str]) -> tuple[int, int]:
    numeric_count = sum(1 for key in kpis if key in NUMERIC_KPI_FIELDS)
    metadata_count = sum(1 for value in metadata.values() if value)
    return numeric_count, metadata_count


def load_source_rows(
    csv_paths: list[Path],
    company_map: dict[str, dict[str, str]],
) -> list[SourceRow]:
    rows: list[SourceRow] = []
    for path in csv_paths:
        with path.open("r", encoding="utf-8-sig", newline="", errors="replace") as handle:
            for raw_row in csv.DictReader(handle):
                raw_company = clean_text(raw_row.get("company"))
                mapped = company_map.get(normalize_source_key(raw_company), {})
                company = clean_text(mapped.get("company")) or title_company_name(raw_company)
                key = canonical_company_key(company)
                if not company or not key:
                    continue
                rows.append(
                    SourceRow(
                        row=raw_row,
                        source_file=str(path),
                        source_dataset=source_label(path),
                        company=company,
                        company_key=key,
                        nse_symbol=clean_text(mapped.get("nse_symbol")),
                    )
                )
    return rows


def build_metadata(source: SourceRow) -> dict[str, str]:
    row = source.row
    metadata = {
        "company": source.company,
        "reporting_year": normalize_reporting_year(row.get("reporting_year")),
        "sector": clean_text(row.get("meta_sector")),
        "market_cap": clean_text(row.get("meta_market_cap")),
        "framework": clean_text(row.get("meta_framework_used")),
        "brsr_version": clean_text(row.get("meta_brsr_version")),
        "assurance": clean_text(row.get("meta_assurance_type")),
        "geography": clean_text(row.get("meta_geography")),
        "nse_symbol": source.nse_symbol,
    }
    cleaned = {
        "company": source.company,
        "reporting_year": metadata["reporting_year"],
    }
    cleaned.update(
        {
            key: value
            for key, value in metadata.items()
            if key not in {"company", "reporting_year"}
            and value
            and not is_missing(value)
        }
    )
    return cleaned


def deduplicate(rows: list[SourceRow]) -> tuple[list[SourceRow], dict[str, Any]]:
    grouped: dict[str, list[SourceRow]] = defaultdict(list)
    for row in rows:
        grouped[row.company_key].append(row)

    selected: list[SourceRow] = []
    duplicate_audit: list[dict[str, Any]] = []
    for company_key, candidates in grouped.items():
        ranked = sorted(
            candidates,
            key=lambda item: row_quality(clean_kpis(item.row), build_metadata(item)),
            reverse=True,
        )
        winner = ranked[0]
        display_name = max(
            (item.company for item in candidates),
            key=lambda name: (
                "limited" in name.lower(),
                " " in name,
                not name.isupper(),
                len(name),
            ),
        )
        winner.company = display_name
        selected.append(winner)
        if len(ranked) > 1:
            duplicate_audit.append(
                {
                    "company_key": company_key,
                    "selected_company": winner.company,
                    "selected_source": winner.source_file,
                    "candidate_count": len(ranked),
                    "discarded_sources": [item.source_file for item in ranked[1:]],
                }
            )
    return selected, {
        "source_rows": len(rows),
        "unique_company_keys": len(grouped),
        "duplicate_company_groups": len(duplicate_audit),
        "duplicate_rows_removed": len(rows) - len(selected),
        "duplicate_audit": duplicate_audit,
    }


def movement_sentence(
    metric: str,
    current: Any,
    previous: Any,
    yoy: Any,
    unit: str,
) -> tuple[str | None, list[dict[str, str]]]:
    current_text = format_number(current)
    previous_text = format_number(previous)
    if current_text is None:
        return None, []

    facts = [{"field": metric, "value": current_text, "unit": unit, "kind": "current"}]
    if previous_text is None:
        return f"{metric} stood at {current_text} {unit}.", facts

    facts.append({"field": metric, "value": previous_text, "unit": unit, "kind": "previous"})
    sentence = f"{metric} stood at {current_text} {unit}, compared with {previous_text} {unit} in the previous year."
    yoy_number = as_number(yoy)
    yoy_text = format_percent(yoy_number)
    if yoy_number is not None and yoy_text:
        direction = "reduced" if yoy_number > 0 else "increased" if yoy_number < 0 else "remained unchanged"
        if direction == "remained unchanged":
            sentence += f" {metric} remained unchanged year-on-year."
        else:
            sentence += f" {metric} {direction} by {yoy_text} year-on-year."
        facts.append(
            {
                "field": metric,
                "value": yoy_text,
                "unit": "%",
                "kind": "yoy",
                "direction": direction,
            }
        )
    return sentence, facts


def append_simple_fact(
    sentences: list[str],
    facts: list[dict[str, str]],
    label: str,
    value: Any,
    unit: str,
    sentence_template: str,
) -> None:
    formatted = format_number(value)
    if formatted is None:
        return
    sentences.append(sentence_template.format(value=formatted, unit=unit))
    facts.append({"field": label, "value": formatted, "unit": unit, "kind": "current"})


def build_target(metadata: dict[str, str], kpis: dict[str, Any]) -> tuple[str, list[dict[str, str]]]:
    company = metadata["company"]
    year = metadata.get("reporting_year", "the reporting year")
    coverage = []
    if any(key.startswith("kpi_scope") for key in kpis):
        coverage.append("greenhouse gas emissions")
    if any("energy" in key for key in kpis):
        coverage.append("energy")
    if any("water" in key for key in kpis):
        coverage.append("water")
    if any("waste" in key for key in kpis):
        coverage.append("waste")
    if any(key in kpis for key in ("kpi_female_employee_percent", "kpi_women_on_board_percent")):
        coverage.append("workforce and board diversity")

    coverage_text = ", ".join(coverage[:-1]) + (" and " + coverage[-1] if len(coverage) > 1 else coverage[0] if coverage else "")
    first = (
        f"{company} disclosed selected ESG performance indicators for {year} "
        "based on the available structured BRSR KPI data."
    )
    if coverage_text:
        first += f" The available indicators cover {coverage_text}."
    paragraphs = [first]
    facts: list[dict[str, str]] = []

    emissions = []
    for label, prefix in (
        ("Scope 1 emissions", "scope1"),
        ("Scope 2 emissions", "scope2"),
        ("Scope 3 emissions", "scope3"),
        ("Combined Scope 1 and Scope 2 emissions", "scope1_scope2_total"),
    ):
        current_field, previous_field, yoy_field = YOY_SPECS[prefix]
        sentence, new_facts = movement_sentence(
            label,
            kpis.get(current_field),
            kpis.get(previous_field),
            kpis.get(yoy_field),
            "tCO2e",
        )
        if sentence:
            emissions.append(sentence)
            facts.extend(new_facts)
    if emissions:
        paragraphs.append(" ".join(emissions))

    resources: list[str] = []
    append_simple_fact(
        resources,
        facts,
        "Total energy consumption",
        kpis.get("kpi_total_energy_consumption_gj"),
        "GJ",
        "Total energy consumption was {value} {unit}.",
    )
    append_simple_fact(
        resources,
        facts,
        "Renewable energy consumption",
        kpis.get("kpi_renewable_energy_consumption_gj"),
        "GJ",
        "Renewable energy consumption was {value} {unit}.",
    )
    renewable_percent = format_percent(kpis.get("kpi_renewable_energy_percent"))
    if renewable_percent:
        resources.append(f"Renewable energy represented {renewable_percent} of the disclosed energy mix.")
        facts.append({"field": "Renewable energy share", "value": renewable_percent, "unit": "%", "kind": "current"})

    energy_intensity_unit = clean_text(kpis.get("kpi_energy_intensity_unit")) or "in the disclosed unit"
    sentence, new_facts = movement_sentence(
        "Energy intensity",
        kpis.get("kpi_energy_intensity_current"),
        kpis.get("kpi_energy_intensity_previous"),
        kpis.get("kpi_energy_intensity_yoy_reduction_percent"),
        energy_intensity_unit,
    )
    if sentence:
        resources.append(sentence)
        facts.extend(new_facts)

    for label, prefix in (
        ("Water consumption", "water_consumption"),
        ("Water withdrawal", "water_withdrawal"),
    ):
        current_field, previous_field, yoy_field = YOY_SPECS[prefix]
        sentence, new_facts = movement_sentence(
            label,
            kpis.get(current_field),
            kpis.get(previous_field),
            kpis.get(yoy_field),
            "KL",
        )
        if sentence:
            resources.append(sentence)
            facts.extend(new_facts)

    waste_unit = clean_text(kpis.get("kpi_waste_recycled_unit")) or "tonnes"
    total_waste_current = format_number(kpis.get("kpi_total_waste_generated_current"))
    total_waste_previous = format_number(kpis.get("kpi_total_waste_generated_previous"))
    if total_waste_current:
        text = f"Total waste generated was {total_waste_current} {waste_unit}"
        facts.append({"field": "Total waste generated", "value": total_waste_current, "unit": waste_unit, "kind": "current"})
        if total_waste_previous:
            text += f", compared with {total_waste_previous} {waste_unit} in the previous year"
            facts.append({"field": "Total waste generated", "value": total_waste_previous, "unit": waste_unit, "kind": "previous"})
        resources.append(text + ".")

    recycled_current = format_number(kpis.get("kpi_waste_recycled_current"))
    recycled_previous = format_number(kpis.get("kpi_waste_recycled_previous"))
    if recycled_current:
        text = f"Waste recycled or recovered was {recycled_current} {waste_unit}"
        facts.append({"field": "Waste recycled or recovered", "value": recycled_current, "unit": waste_unit, "kind": "current"})
        if recycled_previous:
            text += f", compared with {recycled_previous} {waste_unit} in the previous year"
            facts.append({"field": "Waste recycled or recovered", "value": recycled_previous, "unit": waste_unit, "kind": "previous"})
        resources.append(text + ".")

    recycled_percent = format_percent(kpis.get("kpi_waste_recycled_percent"))
    if recycled_percent:
        resources.append(f"The structured KPI data reports a waste recycling or recovery share of {recycled_percent}.")
        facts.append({"field": "Waste recycling or recovery share", "value": recycled_percent, "unit": "%", "kind": "current"})

    if resources:
        paragraphs.append(" ".join(resources))

    social: list[str] = []
    female_percent = format_percent(kpis.get("kpi_female_employee_percent"))
    if female_percent:
        social.append(f"Female employees represented {female_percent} of the disclosed workforce.")
        facts.append({"field": "Female employees", "value": female_percent, "unit": "%", "kind": "current"})
    board_percent = format_percent(kpis.get("kpi_women_on_board_percent"))
    if board_percent:
        social.append(f"Women represented {board_percent} of the disclosed Board composition.")
        facts.append({"field": "Women on the Board", "value": board_percent, "unit": "%", "kind": "current"})
    if social:
        paragraphs.append(" ".join(social))

    closing: list[str] = []
    net_zero_year = clean_text(kpis.get("kpi_net_zero_target_year"))
    if net_zero_year:
        closing.append(f"The structured KPI data records a net-zero target year of {net_zero_year}.")
        facts.append({"field": "Net-zero target year", "value": net_zero_year, "unit": "year", "kind": "target"})
    if metadata.get("assurance"):
        closing.append(f"The reported assurance basis is {metadata['assurance']}.")
    closing.append(
        "This summary is limited to the supplied structured KPIs and metadata and "
        "does not add unsupported policies, initiatives, awards, committees, "
        "certifications, or external-framework claims."
    )
    paragraphs.append(" ".join(closing))

    return "\n\n".join(paragraphs), facts


def build_prompt(metadata: dict[str, str], kpis: dict[str, Any]) -> str:
    metadata_lines = [
        f"- {key.replace('_', ' ').title()}: {value}"
        for key, value in metadata.items()
    ]
    kpi_lines = []
    for field in NUMERIC_KPI_FIELDS + TEXT_KPI_FIELDS:
        if field not in kpis:
            continue
        value = format_number(kpis[field]) if field in NUMERIC_KPI_FIELDS else clean_text(kpis[field])
        if value is not None:
            kpi_lines.append(f"- {FIELD_LABELS[field]}: {value}")
    return (
        "Task:\n"
        "Write one professional ESG narrative summary that can be used as the "
        "starting point for a company report. Cover every supplied KPI, compare "
        "current and previous values where available, and do not add unsupported claims.\n\n"
        "Company metadata:\n"
        + "\n".join(metadata_lines)
        + "\n\nKPI data:\n"
        + ("\n".join(kpi_lines) if kpi_lines else "No valid structured KPI values supplied.")
    )


def stable_split(company_key: str) -> str:
    bucket = int(hashlib.sha256(company_key.encode("utf-8")).hexdigest()[:8], 16) % 100
    if bucket < 80:
        return "train"
    if bucket < 90:
        return "validation"
    return "test"


def make_example(source: SourceRow) -> dict[str, Any] | None:
    metadata = build_metadata(source)
    if metadata.get("reporting_year") not in {"FY 2024-25", "FY 2025-26"}:
        return None
    kpis = clean_kpis(source.row)
    if len(kpis) < 2:
        return None
    target, expected_facts = build_target(metadata, kpis)
    if not expected_facts:
        return None
    prompt = build_prompt(metadata, kpis)
    split = stable_split(source.company_key)
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": target},
        ],
        "prompt": prompt,
        "target_summary": target,
        "company": metadata["company"],
        "company_key": source.company_key,
        "reporting_year": metadata.get("reporting_year"),
        "source_dataset": source.source_dataset,
        "source_file": source.source_file,
        "split": split,
        "metadata": metadata,
        "kpis": kpis,
        "expected_facts": expected_facts,
        "num_kpis": len(kpis),
    }


def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_quality_csv(path: Path, examples: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "company",
        "company_key",
        "split",
        "reporting_year",
        "source_dataset",
        "num_kpis",
        "num_expected_facts",
        "prompt_chars",
        "target_chars",
    ]
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for example in examples:
            writer.writerow(
                {
                    "company": example["company"],
                    "company_key": example["company_key"],
                    "split": example["split"],
                    "reporting_year": example["reporting_year"],
                    "source_dataset": example["source_dataset"],
                    "num_kpis": example["num_kpis"],
                    "num_expected_facts": len(example["expected_facts"]),
                    "prompt_chars": len(example["prompt"]),
                    "target_chars": len(example["target_summary"]),
                }
            )


def build_dataset(
    csv_paths: list[Path],
    output_dir: Path,
    company_map_path: Path | None = None,
) -> dict[str, Any]:
    company_map = load_company_map(company_map_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    if company_map:
        with (output_dir / "company_name_map.json").open("w", encoding="utf-8") as handle:
            json.dump(company_map, handle, indent=2, ensure_ascii=False)
    source_rows = load_source_rows(csv_paths, company_map)
    selected_rows, dedupe_manifest = deduplicate(source_rows)
    examples = [example for row in selected_rows if (example := make_example(row))]
    examples.sort(key=lambda item: (item["split"], item["company_key"]))

    split_rows = {
        split: [row for row in examples if row["split"] == split]
        for split in ("train", "validation", "test")
    }
    write_jsonl(output_dir / "kpi_summary_all.jsonl", examples)
    for split, rows in split_rows.items():
        write_jsonl(output_dir / f"kpi_summary_{split}.jsonl", rows)
    write_quality_csv(output_dir / "dataset_quality.csv", examples)

    company_sets = {
        split: {row["company_key"] for row in rows}
        for split, rows in split_rows.items()
    }
    overlap = {
        "train_validation": sorted(company_sets["train"] & company_sets["validation"]),
        "train_test": sorted(company_sets["train"] & company_sets["test"]),
        "validation_test": sorted(company_sets["validation"] & company_sets["test"]),
    }
    manifest = {
        "task": "company_metadata_and_kpis_to_grounded_esg_summary",
        "source_files": [str(path) for path in csv_paths],
        "company_map_source": company_map_path.name if company_map_path else None,
        "portable_company_map": "company_name_map.json" if company_map else None,
        "excluded_inputs": [
            "llm_training_summary",
            "intent labels and counts",
            "section classification flags and scores",
            "KPI evidence text",
            "SusGen",
            "annual reports 2022-23",
        ],
        "source_rows": dedupe_manifest["source_rows"],
        "deduplicated_company_rows": len(selected_rows),
        "examples_after_quality_filter": len(examples),
        "split_counts": {split: len(rows) for split, rows in split_rows.items()},
        "company_overlap": overlap,
        "kpi_count_distribution": dict(Counter(row["num_kpis"] for row in examples)),
        "source_distribution": dict(Counter(row["source_dataset"] for row in examples)),
        "reporting_year_distribution": dict(Counter(row["reporting_year"] for row in examples)),
        "deduplication": dedupe_manifest,
    }
    with (output_dir / "dataset_manifest.json").open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)

    sample_rows = []
    for split in ("train", "validation", "test"):
        sample_rows.extend(split_rows[split][:1])
    with (output_dir / "sample_summaries.json").open("w", encoding="utf-8") as handle:
        json.dump(sample_rows, handle, indent=2, ensure_ascii=False)
    return manifest


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "paths",
        nargs="+",
        help="The BRSR 2024-25 and 2025-26 PRD master CSV files or their directory.",
    )
    parser.add_argument(
        "--company-map-placeholder-jsonl",
        type=Path,
        default=None,
        help="Optional placeholder JSONL used only to recover clean company names for raw 2025-26 filenames.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("data/kpi_summary"),
    )
    args = parser.parse_args()

    csv_paths = find_csvs(args.paths)
    if not csv_paths:
        raise FileNotFoundError("No BRSR 2024-25 or 2025-26 PRD master CSV files were found.")
    print("Using source files:")
    for path in csv_paths:
        print("-", path)
    manifest = build_dataset(
        csv_paths,
        args.output_dir,
        args.company_map_placeholder_jsonl,
    )
    print(json.dumps(
        {
            "examples": manifest["examples_after_quality_filter"],
            "splits": manifest["split_counts"],
            "overlap": manifest["company_overlap"],
            "output_dir": str(args.output_dir),
        },
        indent=2,
    ))


In [ ]:
# Cell 4: Locate prebuilt data or build it from the two PRD master CSVs
SEARCH_ROOTS = [Path("/kaggle/input")] if IS_KAGGLE else [Path.cwd(), Path.cwd() / "data"]

def find_files(filename):
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(root.rglob(filename))
    return sorted(set(matches))

prebuilt_train = find_files("kpi_summary_train.jsonl")
if not prebuilt_train:
    dataset_archives = find_files("CarbonTatvaAI_KPI_Summary_Dataset.zip")
    if dataset_archives:
        shutil.unpack_archive(str(dataset_archives[0]), str(DATA_OUTPUT_DIR))
        prebuilt_train = sorted(DATA_OUTPUT_DIR.rglob("kpi_summary_train.jsonl"))
        print("Extracted dataset bundle:", dataset_archives[0])

if prebuilt_train:
    DATA_DIR = prebuilt_train[0].parent
    print("Using prebuilt dataset:", DATA_DIR)
else:
    csv_24 = find_files("esg_prd_master_dataset_24_25.csv")
    csv_25 = find_files("esg_prd_master_dataset_25-26.csv")
    if not csv_24 or not csv_25:
        raise FileNotFoundError(
            "Upload either the kpi_summary dataset bundle or both PRD master CSV files."
        )

    compact_maps = find_files("company_name_map.json")
    placeholder_maps = find_files("llama_esg_placeholder_all.jsonl")
    company_map = compact_maps[0] if compact_maps else placeholder_maps[0] if placeholder_maps else None

    manifest = build_dataset(
        [csv_24[0], csv_25[0]],
        DATA_OUTPUT_DIR,
        company_map,
    )
    DATA_DIR = DATA_OUTPUT_DIR
    print(json.dumps(manifest["split_counts"], indent=2))

TRAIN_JSONL = DATA_DIR / "kpi_summary_train.jsonl"
VALID_JSONL = DATA_DIR / "kpi_summary_validation.jsonl"
TEST_JSONL = DATA_DIR / "kpi_summary_test.jsonl"

for path in (TRAIN_JSONL, VALID_JSONL, TEST_JSONL):
    if not path.exists():
        raise FileNotFoundError(path)
    print(path.name, round(path.stat().st_size / 1024 / 1024, 2), "MB")


In [ ]:
# Cell 5: Dataset audit
from collections import Counter

def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_rows = read_jsonl(TRAIN_JSONL)
valid_rows = read_jsonl(VALID_JSONL)
test_rows = read_jsonl(TEST_JSONL)

train_companies = {row["company_key"] for row in train_rows}
valid_companies = {row["company_key"] for row in valid_rows}
test_companies = {row["company_key"] for row in test_rows}

assert not train_companies & valid_companies
assert not train_companies & test_companies
assert not valid_companies & test_companies
assert all("Narrative intent distribution" not in row["target_summary"] for row in train_rows)

print("Train / validation / test:", len(train_rows), len(valid_rows), len(test_rows))
print("Company overlap: 0")
print("Reporting years:", Counter(row["reporting_year"] for row in train_rows + valid_rows + test_rows))
print("Median KPI count:", sorted(row["num_kpis"] for row in train_rows)[len(train_rows) // 2])
print("\nExample company:", train_rows[0]["company"])
print(train_rows[0]["target_summary"][:1800])


In [ ]:
# Cell 6: Load the 4-bit base model
import gc
import torch

# Import Unsloth before transformers/trl.
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

torch.cuda.empty_cache()
gc.collect()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_NAME)


In [ ]:
# Cell 7: Load and format the company-disjoint splits
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_JSONL),
        "validation": str(VALID_JSONL),
        "test": str(TEST_JSONL),
    },
)

def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(format_chat, batched=True)
if SMOKE_TEST:
    dataset["train"] = dataset["train"].select(range(min(SMOKE_TRAIN_EXAMPLES, len(dataset["train"]))))

print(dataset)
print(dataset["train"][0]["text"][:1200])


In [ ]:
# Cell 8: Capture base-model outputs for three unseen test companies
def generate_from_messages(messages, max_new_tokens=700):
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
comparison_rows = [test_rows[index] for index in range(min(3, len(test_rows)))]
base_outputs = {}
for row in comparison_rows:
    base_outputs[row["company_key"]] = generate_from_messages(row["messages"][:2])
    print("\nBASE:", row["company"])
    print(base_outputs[row["company_key"]][:900])

with (ARTIFACT_DIR / "base_outputs.json").open("w", encoding="utf-8") as handle:
    json.dump(base_outputs, handle, indent=2, ensure_ascii=False)


In [ ]:
# Cell 9: Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()


In [ ]:
# Cell 10: Train for one epoch with 15-minute resumable checkpoints
import inspect
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

class TimeBasedSaveCallback(TrainerCallback):
    def __init__(self, minutes=15):
        self.interval_seconds = minutes * 60
        self.last_save_time = time.time()
        self.last_save_step = 0

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if state.global_step > self.last_save_step and now - self.last_save_time >= self.interval_seconds:
            print(f"\nSaving 15-minute checkpoint at step {state.global_step}")
            control.should_save = True
            self.last_save_time = now
            self.last_save_step = state.global_step
        return control

config_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "optim": "adamw_8bit",
    "logging_steps": 5,
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": 3,
    "report_to": "none",
    "seed": SEED,
    "dataset_text_field": "text",
    "dataset_num_proc": 1,
    "packing": False,
    "padding_free": False,
    "gradient_checkpointing": True,
}
if SMOKE_TEST:
    config_kwargs["max_steps"] = SMOKE_MAX_STEPS

supported_config = set(inspect.signature(SFTConfig.__init__).parameters)
if "max_length" in supported_config:
    config_kwargs["max_length"] = MAX_SEQ_LENGTH
elif "max_seq_length" in supported_config:
    config_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
config_kwargs = {key: value for key, value in config_kwargs.items() if key in supported_config}
sft_config = SFTConfig(**config_kwargs)

trainer_kwargs = {
    "model": model,
    "args": sft_config,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
    "processing_class": tokenizer,
    "tokenizer": tokenizer,
}
supported_trainer = set(inspect.signature(SFTTrainer.__init__).parameters)
trainer_kwargs = {key: value for key, value in trainer_kwargs.items() if key in supported_trainer}
trainer = SFTTrainer(**trainer_kwargs)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.add_callback(TimeBasedSaveCallback(SAVE_EVERY_MINUTES))

# If a checkpoint dataset was uploaded after a Kaggle restart, import the
# highest-step checkpoint before looking for a local resume point.
if IS_KAGGLE and not list(CHECKPOINT_DIR.glob("checkpoint-*")):
    external_checkpoints = sorted(
        Path("/kaggle/input").rglob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]) if path.name.split("-")[-1].isdigit() else -1,
    )
    if external_checkpoints:
        source_checkpoint = external_checkpoints[-1]
        destination_checkpoint = CHECKPOINT_DIR / source_checkpoint.name
        shutil.copytree(source_checkpoint, destination_checkpoint, dirs_exist_ok=True)
        print("Imported uploaded checkpoint:", source_checkpoint)

checkpoint_candidates = sorted(
    CHECKPOINT_DIR.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)
resume_checkpoint = str(checkpoint_candidates[-1]) if checkpoint_candidates else None
if resume_checkpoint:
    print("Resuming from:", resume_checkpoint)

FastLanguageModel.for_training(model)
trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
print(trainer_stats)


In [ ]:
# Cell 11: Save the final LoRA adapter
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

with (ARTIFACT_DIR / "training_stats.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "train_runtime": getattr(trainer_stats, "metrics", {}).get("train_runtime"),
            "train_loss": getattr(trainer_stats, "metrics", {}).get("train_loss"),
            "global_step": trainer.state.global_step,
            "epochs": NUM_EPOCHS,
        },
        handle,
        indent=2,
    )
print("Saved adapter:", FINAL_ADAPTER_DIR)


In [ ]:
# Cell 12: Evaluate factual KPI fidelity on unseen companies
import re
from statistics import mean
from rouge_score import rouge_scorer

FastLanguageModel.for_inference(model)

def numeric_value(text):
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", str(text))
    return float(match.group(0).replace(",", "")) if match else None

def extract_numbers(text):
    return [
        float(value.replace(",", ""))
        for value in re.findall(r"(?<![A-Za-z])(-?\d[\d,]*(?:\.\d+)?)", text or "")
    ]

def close_number(left, right):
    return abs(left - right) <= max(0.02, abs(right) * 0.002)

def score_facts(generated, row):
    generated_numbers = extract_numbers(generated)
    expected_numbers = []
    matched = 0
    yoy_total = 0
    yoy_correct = 0
    for fact in row["expected_facts"]:
        expected = numeric_value(fact["value"])
        if expected is None:
            continue
        expected_numbers.append(expected)
        number_found = any(close_number(number, expected) for number in generated_numbers)
        matched += int(number_found)
        if fact.get("kind") == "yoy":
            yoy_total += 1
            direction = fact.get("direction", "")
            yoy_correct += int(number_found and direction in generated.lower())

    allowed_numbers = expected_numbers + [1, 2, 3]
    year = row.get("reporting_year") or ""
    allowed_numbers.extend(extract_numbers(year))
    unsupported = [
        number
        for number in generated_numbers
        if not any(close_number(number, allowed) for allowed in allowed_numbers)
    ]
    unsupported_claim_terms = [
        "science-based target",
        "carbon offset",
        "tcfd aligned",
        "sustainability committee",
        "award-winning",
        "certified by",
    ]
    unsupported_claims = [
        term
        for term in unsupported_claim_terms
        if term in generated.lower() and term not in row["prompt"].lower()
    ]
    return {
        "fact_coverage": matched / max(len(expected_numbers), 1),
        "direction_accuracy": yoy_correct / max(yoy_total, 1),
        "unsupported_number_rate": len(unsupported) / max(len(generated_numbers), 1),
        "unsupported_claim_count": len(unsupported_claims),
        "unsupported_numbers": unsupported,
        "unsupported_claims": unsupported_claims,
    }

eval_rows = test_rows[: min(MAX_EVAL_SAMPLES, len(test_rows))]
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
evaluation = []
for index, row in enumerate(eval_rows, 1):
    generated = generate_from_messages(row["messages"][:2])
    metrics = score_facts(generated, row)
    metrics["rouge_l"] = scorer.score(row["target_summary"], generated)["rougeL"].fmeasure
    evaluation.append({
        "company": row["company"],
        "company_key": row["company_key"],
        "generated_summary": generated,
        "reference_summary": row["target_summary"],
        **metrics,
    })
    if index % 5 == 0:
        print(f"Evaluated {index}/{len(eval_rows)}")

summary_metrics = {
    "samples": len(evaluation),
    "kpi_fact_coverage": mean(item["fact_coverage"] for item in evaluation),
    "yoy_direction_accuracy": mean(item["direction_accuracy"] for item in evaluation),
    "unsupported_number_rate": mean(item["unsupported_number_rate"] for item in evaluation),
    "unsupported_claims_per_sample": mean(item["unsupported_claim_count"] for item in evaluation),
    "rouge_l": mean(item["rouge_l"] for item in evaluation),
}

if RUN_BERTSCORE:
    if importlib.util.find_spec("bert_score") is None:
        pip_install("bert-score")
    from bert_score import score as bert_score
    _, _, f1 = bert_score(
        [item["generated_summary"] for item in evaluation],
        [item["reference_summary"] for item in evaluation],
        lang="en",
        model_type="distilbert-base-uncased",
        num_layers=5,
        device="cpu",
        verbose=False,
    )
    summary_metrics["bertscore_f1"] = float(f1.mean())

with (ARTIFACT_DIR / "evaluation_details.json").open("w", encoding="utf-8") as handle:
    json.dump(evaluation, handle, indent=2, ensure_ascii=False)
with (ARTIFACT_DIR / "evaluation_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(summary_metrics, handle, indent=2)

print(json.dumps(summary_metrics, indent=2))


In [ ]:
# Cell 13: Save three base-vs-fine-tuned held-out comparisons
comparisons = []
for row in comparison_rows:
    tuned = generate_from_messages(row["messages"][:2])
    comparisons.append({
        "company": row["company"],
        "prompt": row["prompt"],
        "base_model": base_outputs[row["company_key"]],
        "fine_tuned_model": tuned,
        "reference": row["target_summary"],
        "fine_tuned_factual_metrics": score_facts(tuned, row),
    })

with (ARTIFACT_DIR / "heldout_comparisons.json").open("w", encoding="utf-8") as handle:
    json.dump(comparisons, handle, indent=2, ensure_ascii=False)

for item in comparisons:
    print("\n" + "=" * 100)
    print(item["company"])
    print("\nBASE MODEL:\n", item["base_model"][:1200])
    print("\nFINE-TUNED MODEL:\n", item["fine_tuned_model"][:1200])
    print("\nREFERENCE:\n", item["reference"][:1200])


In [ ]:
# Cell 14: Manual inference for a new company
# Replace these values with a company's structured metadata and KPI data.
MANUAL_METADATA = {
    "company": "Example Industries Limited",
    "reporting_year": "FY 2024-25",
    "sector": "Industrials",
    "geography": "India",
}

MANUAL_KPIS = {
    "kpi_scope1_emissions_tco2e_current": 800,
    "kpi_scope1_emissions_tco2e_previous": 1000,
    "kpi_scope2_emissions_tco2e_current": 450,
    "kpi_scope2_emissions_tco2e_previous": 500,
    "kpi_total_energy_consumption_gj": 25000,
    "kpi_water_consumption_kl_current": 12000,
    "kpi_water_consumption_kl_previous": 14000,
}

manual_kpis = clean_kpis(MANUAL_KPIS)
manual_prompt = build_prompt(MANUAL_METADATA, manual_kpis)
manual_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": manual_prompt},
]
manual_summary = generate_from_messages(manual_messages)
print(manual_summary)


In [ ]:
# Cell 15: Export the trained model for Ollama deployment
import zipfile

OLLAMA_EXPORT_DIR = WORK_DIR / "ollama_export"
if OLLAMA_EXPORT_DIR.exists():
    shutil.rmtree(OLLAMA_EXPORT_DIR)
OLLAMA_EXPORT_DIR.mkdir(parents=True)

print("Merging the LoRA adapter and exporting Q4_K_M GGUF.")
print("This export can take 20-40 minutes and needs several GB of free disk space.")
model.save_pretrained_gguf(
    str(OLLAMA_EXPORT_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_candidates = list(OLLAMA_EXPORT_DIR.rglob("*.gguf"))
if not gguf_candidates:
    raise RuntimeError("Unsloth export completed without producing a GGUF file.")

exported_gguf = max(gguf_candidates, key=lambda path: path.stat().st_size)
ollama_gguf = OLLAMA_EXPORT_DIR / "carbontatva-q4_k_m.gguf"
if exported_gguf.resolve() != ollama_gguf.resolve():
    if ollama_gguf.exists():
        ollama_gguf.unlink()
    exported_gguf.replace(ollama_gguf)

modelfile = '''FROM ./carbontatva-q4_k_m.gguf

PARAMETER temperature 0
PARAMETER num_ctx 4096
PARAMETER repeat_penalty 1.05

SYSTEM """You are CarbonTatvaAI, an ESG reporting analyst. Write a concise, factual ESG narrative summary using only the supplied company metadata and KPI data. Mention all material KPI values that are provided, preserve units, and describe year-on-year movement accurately. Do not invent policies, initiatives, awards, targets, committees, certifications, framework alignment, or explanations that are absent from the input. The summary is a drafting aid, not a complete statutory BRSR report."""
'''
(OLLAMA_EXPORT_DIR / "Modelfile").write_text(modelfile, encoding="utf-8")

example_input = {
    "metadata": MANUAL_METADATA,
    "kpis": MANUAL_KPIS,
}
(OLLAMA_EXPORT_DIR / "example_input.json").write_text(
    json.dumps(example_input, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

client_source = '#!/usr/bin/env python3\n"""Send company metadata and KPI JSON to the local CarbonTatvaAI Ollama model."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom typing import Any\n\n\nDEFAULT_MODEL = "carbontatva"\nDEFAULT_URL = "http://localhost:11434/api/generate"\n\nSYSTEM_PROMPT = (\n    "You are CarbonTatvaAI, an ESG reporting analyst. Write a concise, factual "\n    "ESG narrative summary using only the supplied company metadata and KPI "\n    "data. Mention all material KPI values that are provided, preserve units, "\n    "and describe year-on-year movement accurately. Do not invent policies, "\n    "initiatives, awards, targets, committees, certifications, framework "\n    "alignment, or explanations that are absent from the input. The summary is "\n    "a drafting aid, not a complete statutory BRSR report."\n)\n\n\ndef label(key: str) -> str:\n    return key.removeprefix("kpi_").replace("_", " ").strip().title()\n\n\ndef build_prompt(metadata: dict[str, Any], kpis: dict[str, Any]) -> str:\n    metadata_lines = [\n        f"- {label(key)}: {value}"\n        for key, value in metadata.items()\n        if value is not None and str(value).strip()\n    ]\n    kpi_lines = [\n        f"- {label(key)}: {value}"\n        for key, value in kpis.items()\n        if value is not None and str(value).strip()\n    ]\n    return (\n        "Task:\\n"\n        "Write one professional ESG narrative summary that can be used as the "\n        "starting point for a company report. Cover every supplied KPI, compare "\n        "current and previous values where available, and do not add unsupported claims.\\n\\n"\n        "Company metadata:\\n"\n        + "\\n".join(metadata_lines)\n        + "\\n\\nKPI data:\\n"\n        + "\\n".join(kpi_lines)\n    )\n\n\ndef load_input(path: Path) -> tuple[dict[str, Any], dict[str, Any]]:\n    payload = json.loads(path.read_text(encoding="utf-8"))\n    metadata = payload.get("metadata")\n    kpis = payload.get("kpis")\n    if not isinstance(metadata, dict) or not isinstance(kpis, dict):\n        raise ValueError("Input JSON must contain object fields named \'metadata\' and \'kpis\'.")\n    if not str(metadata.get("company", "")).strip():\n        raise ValueError("metadata.company is required.")\n    if len(kpis) < 2:\n        raise ValueError("At least two KPI fields are required.")\n    return metadata, kpis\n\n\ndef generate(model: str, url: str, metadata: dict[str, Any], kpis: dict[str, Any]) -> str:\n    request_body = json.dumps(\n        {\n            "model": model,\n            "system": SYSTEM_PROMPT,\n            "prompt": build_prompt(metadata, kpis),\n            "stream": False,\n            "options": {\n                "temperature": 0,\n                "repeat_penalty": 1.05,\n                "num_ctx": 4096,\n            },\n        }\n    ).encode("utf-8")\n    request = urllib.request.Request(\n        url,\n        data=request_body,\n        headers={"Content-Type": "application/json"},\n        method="POST",\n    )\n    try:\n        with urllib.request.urlopen(request, timeout=600) as response:\n            result = json.loads(response.read().decode("utf-8"))\n    except urllib.error.URLError as exc:\n        raise RuntimeError(\n            "Could not reach Ollama. Install/start Ollama and confirm "\n            "`ollama run carbontatva` works first."\n        ) from exc\n    return str(result.get("response", "")).strip()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("input_json", type=Path)\n    parser.add_argument("--model", default=DEFAULT_MODEL)\n    parser.add_argument("--url", default=DEFAULT_URL)\n    parser.add_argument("--output", type=Path, default=Path("generated_esg_summary.txt"))\n    args = parser.parse_args()\n\n    metadata, kpis = load_input(args.input_json)\n    summary = generate(args.model, args.url, metadata, kpis)\n    args.output.write_text(summary + "\\n", encoding="utf-8")\n    print(summary)\n    print(f"\\nSaved: {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n'
(OLLAMA_EXPORT_DIR / "ollama_kpi_client.py").write_text(
    client_source,
    encoding="utf-8",
)

readme = '''CarbonTatvaAI Ollama Deployment

1. Install Ollama from https://ollama.com/download
2. Extract this folder and open a terminal inside it.
3. Create the local model:
   ollama create carbontatva -f Modelfile
4. Test it interactively:
   ollama run carbontatva
5. Generate an ESG summary from KPI JSON:
   python ollama_kpi_client.py example_input.json

The Ollama API is available locally at:
http://localhost:11434/api
'''
(OLLAMA_EXPORT_DIR / "README_OLLAMA.txt").write_text(readme, encoding="utf-8")

ollama_zip = Path(
    "/kaggle/working/CarbonTatvaAI_Ollama.zip"
    if IS_KAGGLE
    else WORK_DIR / "CarbonTatvaAI_Ollama.zip"
)
if ollama_zip.exists():
    ollama_zip.unlink()
with zipfile.ZipFile(ollama_zip, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(OLLAMA_EXPORT_DIR.iterdir()):
        archive.write(path, arcname=path.name)

print("Ollama model folder:", OLLAMA_EXPORT_DIR)
print("Download for deployment:", ollama_zip)
print("GGUF size (GB):", round(ollama_gguf.stat().st_size / 1024**3, 2))


In [ ]:
# Cell 16: Package everything for download
for filename in ("dataset_manifest.json", "dataset_quality.csv", "sample_summaries.json"):
    source = DATA_DIR / filename
    if source.exists():
        shutil.copy2(source, ARTIFACT_DIR / filename)

delivery_dir = WORK_DIR / "delivery"
if delivery_dir.exists():
    shutil.rmtree(delivery_dir)
delivery_dir.mkdir(parents=True)
shutil.copytree(FINAL_ADAPTER_DIR, delivery_dir / "final_lora_adapter")
shutil.copytree(ARTIFACT_DIR, delivery_dir / "artifacts")

zip_path = Path(shutil.make_archive(
    str(Path("/kaggle/working/CarbonTatvaAI_KPI_Summary") if IS_KAGGLE else WORK_DIR / "CarbonTatvaAI_KPI_Summary"),
    "zip",
    delivery_dir,
))
print("Download this file:", zip_path)
print("Download the Ollama deployment separately:", ollama_zip)
print("Checkpoints remain at:", CHECKPOINT_DIR)


## What to report

> I prepared a company-level BRSR dataset where metadata and structured KPIs are the input and a grounded ESG narrative summary is the output. I fine-tuned Llama 3.1 8B using Unsloth QLoRA and evaluated numerical fidelity and KPI coverage on unseen companies.

Use `evaluation_metrics.json` and `heldout_comparisons.json` as the evidence for this statement.

For deployment, download `CarbonTatvaAI_Ollama.zip`, install Ollama locally,
and run `ollama create carbontatva -f Modelfile`.
